# Irish Weather ML Mini-Lab

**Objective:** Predict the temperature in Dublin using historical weather data from Dublin, Galway, and Cork.

## 1. Setup and Data Loading
We load the dataset and sort by time. We also perform a critical step: **Standardization**. Neural networks struggle when features have vastly different ranges (e.g., Temperature 0–25 vs Wind Speed 0–15).

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

# Load Data
CSV_PATH = "../../data/era5_ireland3_t2m_wind_2024.csv"  # Put this file in the same folder as this notebook
try:
    df = pd.read_csv(CSV_PATH, parse_dates=['time'])
    df = df.sort_values('time').reset_index(drop=True)
    print(f"Data loaded: {len(df)} rows")
except FileNotFoundError:
    print("Error: CSV file not found. Please upload 'era5_ireland3_t2m_wind_2024.csv'.")

# Define features
feature_cols = [
    'Dublin_t2m_degC', 'Galway_t2m_degC', 'Cork_t2m_degC',
    'Dublin_wind_speed10m_ms', 'Galway_wind_speed10m_ms', 'Cork_wind_speed10m_ms'
]

# Apply Scaling
scaler = StandardScaler()
df_scaled = df.copy()
df_scaled[feature_cols] = scaler.fit_transform(df[feature_cols])

print("Feature scaling complete. Mean ~0, Variance ~1.")

Data loaded: 8784 rows
Feature scaling complete. Mean ~0, Variance ~1.


## 2. Vectorized Dataset Class
Instead of a loop, we use direct Numpy slicing. This is much faster and cleaner.

- **Input (`x`):** A sequence of weather data of length `seq_length`.
- **Target (`y`):** The *next* hour's temperature in Dublin.

In [3]:
class WeatherDataset(Dataset):
    def __init__(self, data_df, feature_cols, seq_length=24):
        self.seq_length = seq_length
        # Convert to float32 numpy array once for speed
        self.features = data_df[feature_cols].values.astype(np.float32)
        # Target: Dublin Temp is at index 0 in feature_cols
        self.target = self.features[:, 0]

    def __len__(self):
        # We need enough history (seq_length) to make a prediction
        return len(self.features) - self.seq_length

    def __getitem__(self, idx):
        # Slice the window [t : t+seq_len]
        x = self.features[idx : idx + self.seq_length]
        # Predict the NEXT step [t+seq_len]
        y = self.target[idx + self.seq_length]
        return torch.as_tensor(x), torch.as_tensor(y)

# Prepare Splits (80% Train, 20% Test)
split_idx = int(len(df_scaled) * 0.8)
train_ds = WeatherDataset(df_scaled.iloc[:split_idx], feature_cols)
test_ds = WeatherDataset(df_scaled.iloc[split_idx:], feature_cols)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False)

print(f"Train samples: {len(train_ds)}, Test samples: {len(test_ds)}")

Train samples: 7003, Test samples: 1733


## 3. Establishing a Baseline
Before training a complex CNN, we must ask: *"Can we beat a 'dumb' guess?"*
A common baseline for weather is **Persistence**: assuming the temperature next hour is exactly the same as this hour.

If our CNN Loss > Baseline MSE, the model is not learning anything useful.

In [4]:
def get_persistence_mse(dataset):
    preds, targets = [], []
    for i in range(len(dataset)):
        x, y = dataset[i]
        # The last known value of Dublin Temp (index 0) is our prediction
        last_known_temp = x[-1, 0].item()
        preds.append(last_known_temp)
        targets.append(y.item())
    return np.mean((np.array(preds) - np.array(targets))**2)

baseline_mse = get_persistence_mse(test_ds)
print(f"Persistence Baseline MSE: {baseline_mse:.4f}")
print("(Your CNN must get a lower test error than this to be useful!)")

Persistence Baseline MSE: 0.0108
(Your CNN must get a lower test error than this to be useful!)


## 4. The 1D CNN Model
We use a 1D Convolutional Network. Key architectural details:
- **`nn.Sequential`**: Stacks layers cleanly.
- **`permute(0, 2, 1)`**: PyTorch's `Conv1d` expects the input shape `(Batch, Channels, Length)`. Our dataset provides `(Batch, Length, Channels)`, so we must swap dimensions in the `forward` pass.
- **Dimensionality Reduction**: Each `MaxPool1d(2)` cuts the sequence length in half.

In [5]:
class WeatherCNN1D(nn.Module):
    def __init__(self, in_channels, seq_len):
        super().__init__()
        
        # Feature Extractor: Conv -> ReLU -> Pool
        self.features = nn.Sequential(
            nn.Conv1d(in_channels, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(2), # Reduces length by /2
            
            nn.Conv1d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(2)  # Reduces length by /2 (total /4)
        )
        
        # Flatten and Regress
        # Calculate the flattened size: 32 channels * (seq_len / 4)
        flat_size = 32 * (seq_len // 4)
        
        self.regressor = nn.Sequential(
            nn.Flatten(),
            nn.Linear(flat_size, 1)
        )

    def forward(self, x):
        # x shape: [Batch, Length, Channels]
        # Conv1d needs: [Batch, Channels, Length]
        x = x.permute(0, 2, 1)
        
        x = self.features(x)
        return self.regressor(x).squeeze(-1)

## 5. Training Loop
We use MSE Loss (standard for regression) and the Adam optimizer.

In [6]:
# Initialize
model = WeatherCNN1D(in_channels=len(feature_cols), seq_len=24)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.MSELoss()

# Training
EPOCHS = 5
print("Starting Training...")

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0
    
    for x_batch, y_batch in train_loader:
        optimizer.zero_grad()
        output = model(x_batch)
        loss = criterion(output, y_batch)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    
    avg_loss = train_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{EPOCHS} | Train MSE: {avg_loss:.4f}")

# Evaluation
model.eval()
test_loss = 0
with torch.no_grad():
    for x_batch, y_batch in test_loader:
        output = model(x_batch)
        test_loss += criterion(output, y_batch).item()

final_mse = test_loss / len(test_loader)
print(f"\nFinal Test MSE: {final_mse:.4f}")
print(f"Persistence Baseline: {baseline_mse:.4f}")

if final_mse < baseline_mse:
    print("SUCCESS: Model beat the baseline!")
else:
    print("FAIL: Model did not beat the baseline. Try tuning hyperparameters.")

Starting Training...
Epoch 1/5 | Train MSE: 0.1585
Epoch 2/5 | Train MSE: 0.0398
Epoch 3/5 | Train MSE: 0.0240
Epoch 4/5 | Train MSE: 0.0179
Epoch 5/5 | Train MSE: 0.0149

Final Test MSE: 0.0186
Persistence Baseline: 0.0108
FAIL: Model did not beat the baseline. Try tuning hyperparameters.
